# Tokenizer From Scratch (Byte-Level BPE)

> Goal: build a production-style tokenizer with train, encode, decode, and serialization support.
> Dataset for initial training: `wizard_of_oz.txt`.
> This tokenizer is designed to plug directly into your future embedding and attention notebooks.

 Data Pre-processing & Memory Mapping

In [ ]:
import os
import numpy as np
import random
from tqdm import tqdm
from datasets import load_dataset
from pathlib import Path

# --- Configuration ---
DATASET_PATH = r"D:\Dataset" # Your local path
TRAIN_BIN = "train.bin"
VAL_BIN = "val.bin"
VAL_PERCENT = 0.05
EOS_TOKEN_ID = tokenizer.special_to_id["<eos>"] # Assuming tokenizer is loaded

# Load local parquet files
ds = load_dataset("parquet", data_files=[os.path.join(DATASET_PATH, "*.parquet")], split="train")

def process_and_save():
    train_ids = []
    val_ids = []
    
    print(f"Tokenizing {len(ds)} documents...")
    for doc in tqdm(ds):
        # Tokenize document
        tokens = tokenizer.encode(doc["text"], add_bos=False, add_eos=False)
        tokens.append(EOS_TOKEN_ID) # Prevent cross-doc bleed
        
        # Document-level split
        if random.random() < VAL_PERCENT:
            val_ids.extend(tokens)
        else:
            train_ids.extend(tokens)

    # Convert to uint16 (efficient for vocab < 65535)
    train_ids = np.array(train_ids, dtype=np.uint16)
    val_ids = np.array(val_ids, dtype=np.uint16)

    # Save to disk as raw binary
    train_ids.tofile(TRAIN_BIN)
    val_ids.tofile(VAL_BIN)
    
    print(f"Saved! Train: {len(train_ids):,} tokens | Val: {len(val_ids):,} tokens")

process_and_save()


In [7]:
from __future__ import annotations

import json
import re
import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np

In [ ]:
import json
import re
import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

@dataclass
class TokenizerConfig:
    vocab_size: int = 32000
    min_pair_freq: int = 2
    special_tokens: Tuple[str, ...] = ("<pad>", "<bos>", "<eos>", "<unk>")

class BytePairTokenizer:
    """Byte-level BPE tokenizer for production LLM training."""
    
    _word_re = re.compile(r"\s+|[^\s]+")

    def __init__(self, config: Optional[TokenizerConfig] = None):
        self.config = config or TokenizerConfig()
        self.base_vocab_size = 256
        self.special_tokens = list(self.config.special_tokens)
        self.special_to_id: Dict[str, int] = {}
        self.id_to_special: Dict[int, str] = {}
        self.merges: Dict[Tuple[int, int], int] = {}
        self.merges_rank: Dict[Tuple[int, int], int] = {}
        self.token_to_bytes: Dict[int, bytes] = {i: bytes([i]) for i in range(self.base_vocab_size)}
        self._init_special_tokens()

    def _init_special_tokens(self) -> None:
        start = self.base_vocab_size
        for i, tok in enumerate(self.special_tokens):
            tid = start + i
            self.special_to_id[tok] = tid
            self.id_to_special[tid] = tok
            self.token_to_bytes[tid] = tok.encode("utf-8")

    @property
    def vocab_size(self) -> int:
        return len(self.token_to_bytes)

    def _merge_sequence(self, seq: Tuple[int, ...], pair: Tuple[int, int], new_id: int) -> Tuple[int, ...]:
        if len(seq) < 2: return seq
        out, i, a, b, n = [], 0, pair[0], pair[1], len(seq)
        while i < n:
            if i < n - 1 and seq[i] == a and seq[i + 1] == b:
                out.append(new_id)
                i += 2
            else:
                out.append(seq[i])
                i += 1
        return tuple(out)

    def train(self, text: str, verbose: bool = True) -> None:
        if not text: raise ValueError("Empty training text.")
        chunks = self._word_re.findall(text)
        word_freqs = Counter(tuple(chunk.encode("utf-8")) for chunk in chunks)
        max_merges = self.config.vocab_size - (self.base_vocab_size + len(self.special_tokens))
        
        next_token_id = self.base_vocab_size + len(self.special_tokens)
        for merge_step in range(max_merges):
            pair_counts: Counter[Tuple[int, int]] = Counter()
            for symbols, freq in word_freqs.items():
                for i in range(len(symbols) - 1):
                    pair_counts[(symbols[i], symbols[i + 1])] += freq

            if not pair_counts: break
            best_pair, best_freq = pair_counts.most_common(1)[0]
            if best_freq < self.config.min_pair_freq: break

            new_id = next_token_id
            next_token_id += 1
            self.merges[best_pair] = new_id
            self.merges_rank[best_pair] = merge_step
            self.token_to_bytes[new_id] = self.token_to_bytes[best_pair[0]] + self.token_to_bytes[best_pair[1]]

            word_freqs = {self._merge_sequence(sym, best_pair, new_id): f 
                          for sym, f in word_freqs.items()}
            
            if verbose and (merge_step + 1) % 500 == 0:
                print(f"Merged {merge_step + 1}/{max_merges} pairs...")

    def encode(self, text: str, add_bos: bool = False, add_eos: bool = False) -> List[int]:
        if not text: return []
        tokens = [self.special_to_id["<bos>"]] if add_bos and "<bos>" in self.special_to_id else []
        for chunk in self._word_re.findall(text):
            symbols = list(chunk.encode("utf-8"))
            while len(symbols) > 1:
                best_pair, best_rank = None, float("inf")
                for i in range(len(symbols) - 1):
                    pair = (symbols[i], symbols[i + 1])
                    rank = self.merges_rank.get(pair, float("inf"))
                    if rank < best_rank:
                        best_rank, best_pair = rank, pair
                if best_pair is None: break
                merged_token = self.merges[best_pair]
                new_symbols, i = [], 0
                while i < len(symbols):
                    if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best_pair:
                        new_symbols.append(merged_token); i += 2
                    else:
                        new_symbols.append(symbols[i]); i += 1
                symbols = new_symbols
            tokens.extend(symbols)
        if add_eos and "<eos>" in self.special_to_id:
            tokens.append(self.special_to_id["<eos>"])
        return tokens

    def decode(self, token_ids: List[int], skip_special_tokens: bool = False) -> str:
        byte_stream = bytearray()
        for tid in token_ids:
            if skip_special_tokens and tid in self.id_to_special: continue
            token_bytes = self.token_to_bytes.get(tid, self.token_to_bytes.get(self.special_to_id.get("<unk>", 0), b""))
            byte_stream.extend(token_bytes)
        return bytes(byte_stream).decode("utf-8", errors="replace")

    def save(self, path: str | Path) -> None:
        payload = {
            "config": {"vocab_size": self.config.vocab_size, "min_pair_freq": self.config.min_pair_freq, "special_tokens": self.special_tokens},
            "merges": [[a, b, new_id] for (a, b), new_id in self.merges.items()],
        }
        Path(path).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    @classmethod
    def load(cls, path: str | Path) -> "BytePairTokenizer":
        payload = json.loads(Path(path).read_text(encoding="utf-8"))
        t = cls(TokenizerConfig(**payload["config"]))
        t.merges.clear(); t.merges_rank.clear(); t.token_to_bytes = {i: bytes([i]) for i in range(t.base_vocab_size)}
        t._init_special_tokens()
        for rank, (a, b, nid) in enumerate(payload["merges"]):
            pair = (int(a), int(b))
            t.merges[pair], t.merges_rank[pair] = int(nid), rank
            t.token_to_bytes[int(nid)] = t.token_to_bytes[pair[0]] + t.token_to_bytes[pair[1]]
        return t


In [9]:
# Train tokenizer on Wizard of Oz
data_path = Path("..") / "wizard_of_oz.txt"
text = data_path.read_text(encoding="utf-8")

tokenizer = BytePairTokenizer(
    TokenizerConfig(
        vocab_size=2000,
        min_pair_freq=2,
        special_tokens=("<pad>", "<bos>", "<eos>", "<unk>"),
    )
)

tokenizer.train(text, verbose=True)
print(f"Final tokenizer vocab size: {tokenizer.vocab_size}")

Merged 200/1740 pairs...
Merged 400/1740 pairs...
Merged 600/1740 pairs...
Merged 800/1740 pairs...
Merged 1000/1740 pairs...
Merged 1200/1740 pairs...
Merged 1400/1740 pairs...
Merged 1600/1740 pairs...
Training complete | merges=1740 | vocab=2000 | time=20.03s
Final tokenizer vocab size: 2000


In [10]:
# Validate encode/decode quality and token compression
sample = "Dorothy lived in the midst of the great Kansas prairies, with Uncle Henry and Aunt Em."
encoded = tokenizer.encode(sample, add_bos=True, add_eos=True)
decoded = tokenizer.decode(encoded)

raw_bytes = len(sample.encode("utf-8"))
token_count = len(encoded)
compression_ratio = raw_bytes / max(token_count, 1)

print("Original:", sample)
print("Decoded :", decoded)
print("Match   :", sample == decoded.replace("<bos>", "").replace("<eos>", ""))
print(f"Bytes={raw_bytes}, Tokens={token_count}, Bytes/Token={compression_ratio:.3f}")
print("First 40 token ids:", encoded[:40])

Original: Dorothy lived in the midst of the great Kansas prairies, with Uncle Henry and Aunt Em.
Decoded : <bos>Dorothy lived in the midst of the great Kansas prairies, with Uncle Henry and Aunt Em.<eos>
Match   : True
Bytes=86, Tokens=42, Bytes/Token=2.048
First 40 token ids: [257, 355, 32, 1556, 32, 264, 32, 261, 32, 1560, 286, 32, 282, 32, 261, 32, 536, 32, 1467, 32, 112, 345, 302, 751, 44, 32, 325, 32, 1290, 32, 1758, 32, 268, 32, 65, 322, 116, 32, 69, 109]


In [11]:
# Save and reload tokenizer
save_path = Path("bpe_tokenizer_wizard.json")
tokenizer.save(save_path)

reloaded = BytePairTokenizer.load(save_path)
encoded_2 = reloaded.encode(sample, add_bos=True, add_eos=True)
decoded_2 = reloaded.decode(encoded_2)

print("Tokenizer saved to:", save_path.resolve())
print("Encoding stable after reload:", encoded == encoded_2)
print("Decoding stable after reload:", decoded == decoded_2)

Tokenizer saved to: C:\Users\Deepesh\Desktop\Mini_Generative_Pretrained_Transformer\Research\bpe_tokenizer_wizard.json
Encoding stable after reload: True
Decoding stable after reload: True
